In [1]:
import collections
import gc
import itertools
import os
import sys
sys.path.append("/workspace/mta_vision_transformers/")
from collections import OrderedDict
from typing import Any, Callable, Dict, Iterable, List, Set, Tuple

import numpy as np
import einops
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import torch.utils.data
from matplotlib import pyplot as plt
from tensordict import TensorDict
from torch.utils._pytree import tree_flatten

from core.monitor import Monitor
from infrastructure import utils
from infrastructure.settings import DEVICE, OUTPUT_DEVICE, DTYPE
from dataset.construct import ImageDataset
from dataset.library import DATASETS


dataset_name, n_classes = DATASETS["Common"][1]
OUTPUT_DIR = "experiments/attention"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
# Ocean: 901085904
# Rose: 100390212
torch.set_printoptions(linewidth=400, sci_mode=False)

Seed: 1149496617


In [2]:
from modeling.vit_attention import OpenCLIPAttentionViT
from modeling.vit_projection import OpenCLIPProjectionViT
from visualize.base import construct_per_layer_output_dict


# SECTION: Set up model
torch.set_default_device(DEVICE)
model = OpenCLIPProjectionViT({}).to(DEVICE)


# SECTION: Set up monitor
def residual_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return input_ + tree_flatten(output_)[0][0]
    
def input_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return tree_flatten(input_)[0][0]

def mean_attention_matrix_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return torch.mean(einops.rearrange(
        tree_flatten(output_)[0][0],
        "b h n1 n2 -> b n1 n2 h"
    ), dim=-1).to(OUTPUT_DEVICE)

def attention_matrix_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return einops.rearrange(
        tree_flatten(output_)[0][0],
        "b h n1 n2 -> b n1 n2 h"
    ).to(OUTPUT_DEVICE)

def subspace_hook_fn(model_: nn.Module, input_: Any, output_: Any) -> Any:
    return einops.rearrange(
        tree_flatten(output_)[0][0],
        "b h n d -> b n h d"
    ).to(OUTPUT_DEVICE)
    
def get_weight_by_name(name: str) -> Callable[[nn.Module, Any, Any], Any]:
    def hook(model_: nn.Module, input_: Any, output_: Any) -> Any:
        return utils.rgetattr(model_, name).data
    return hook


monitor_config = OrderedDict({
    "model.visual.transformer.resblocks": OrderedDict({
        "": [
            ("layer_input", input_hook_fn),
            ("layer_output", Monitor.default_hook_fn),
        ],
        "attn": {
            # "": "attention_output",
            "return_attn_matrix": [
                ("attention_matrix", attention_matrix_hook_fn),
            ],
            "return_value_subspace": [
                ("value_subspace", subspace_hook_fn)
            ],
        },
        "ln_1": "attention_input",  # "norm1"
        # "ln_2": [
        #     ("intermediate_output", input_hook_fn),
        # ],
        # "mlp": "mlp_output"
    })
})

monitor = Monitor(model, monitor_config)
model_weights_config = OrderedDict({
    "attn": OrderedDict({
        "in_proj_weight": "QKVw",
        "in_proj_bias": "QKVb",
        "out_proj.weight": "out_w",
        "out_proj.bias": "out_b",
    })
})


# SECTION: Set up dataset
batch_size = 50
dataset = ImageDataset(dataset_name, split="train", return_original_image=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator(DEVICE))
original_images, images = next(iter(dataloader))
# original_images, images = torch.flip(original_images, dims=[0]), torch.flip(images, dims=[0])


# SECTION: Run baseline model
per_metric_output_dict = monitor.reset()
with torch.no_grad():
    output = model.forward(images)

per_layer_output_dict: List[TensorDict] = construct_per_layer_output_dict(per_metric_output_dict)
model_weights: List[Dict[str, torch.Tensor]] = [{
    v: resblock.get_parameter(k).data
    for k, v in utils.flatten_nested_dict(model_weights_config).items()
} for resblock in model.get_submodule("model.visual.transformer.resblocks")]

In [3]:
# SECTION: Visualize original images
%matplotlib inline
from modeling.image_features import ImageFeatures
from core.attention_sink import mask_attention_sink
from visualize.base import (
    visualize_images_with_mta,
    visualize_feature_norms_per_image,
    get_rgb_colors,
)

torch.set_default_device(OUTPUT_DEVICE)

# SECTION: Massive token heuristic
detection_layer = 13

artifact_mask: torch.Tensor = torch.load(f"experiments/saved_masks/ranked_AS_mask{batch_size}.pt", map_location=OUTPUT_DEVICE) > 0
mta_masks: Dict[int, torch.Tensor] = {
    "MA": mask_attention_sink(
        per_layer_output_dict[detection_layer]["attention_matrix"],
        max_num_tokens=None, scale=0.4, verbose=True
    ),
    "Artifact": artifact_mask
}
features = ImageFeatures(per_layer_output_dict, mta_masks, "default", DEVICE)
visualize_feature_norms_per_image(features, 12, "layer_output", cmap="gray")

for k, v in mta_masks.items():
    print(f"{k}: {v.sum().item()}/{v.numel()}")


# # SECTION: Visualize images
# for mask in mta_masks.values():
#     visualize_images_with_mta(original_images.to(OUTPUT_DEVICE), mask.to(OUTPUT_DEVICE))

try:
    rgb_assignment
except NameError:
    rgb_fname = "sandbox/rgb_assignment.pt"
    if not os.path.exists(rgb_fname):
        color_layer_idx = 10    # min(mta_masks.keys())
        rgb_assignment = get_rgb_colors(features, color_layer_idx, "layer_output", False)
        torch.save(rgb_assignment, rgb_fname)
    else:
        rgb_assignment = torch.load(rgb_fname, map_location=DEVICE)

# highlight = torch.LongTensor((
#     (1, 5, 4),
#     (4, 15, 8),
# ))
# highlight = torch.load(f"{OUTPUT_DIR}/artifact_indices.pt")

/tmp/ipykernel_1073/656357949.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  artifact_mask: torch.Tensor = torch.load(f"experiments/saved_masks/ranked_AS_mask{batch_si

	Image 0 --- CLS attention: 0.1010647788643837, [0.2791483402252197, 0.19634579122066498, 0.01992952823638916, 0.014997958205640316, 0.014932974241673946]
	Image 1 --- CLS attention: 0.07485419511795044, [0.26360899209976196, 0.19319495558738708, 0.08652181178331375, 0.0187437292188406, 0.015619911253452301]
	Image 2 --- CLS attention: 0.11383508145809174, [0.25385433435440063, 0.22178208827972412, 0.04852898046374321, 0.015684744343161583, 0.008137143217027187]
	Image 3 --- CLS attention: 0.08890458196401596, [0.37577158212661743, 0.0958404541015625, 0.021045606583356857, 0.011884110979735851, 0.008227413520216942]


: 

In [ ]:
%matplotlib inline
from dataclasses import dataclass
from core.attention_sink import mask_attention_sink
from modeling.image_features import ImageFeatures
from visualize.base import (
    VISUALIZED_INDICES,
    construct_per_layer_output_dict,
    visualize_features_per_image,
    visualize_feature_norms_per_image,
)
from visualize.attention import (
    visualize_attention_matrix_per_image,
    visualize_attention_suppression_per_image,
    visualize_incoming_attention_per_image,
)
from visualize.projections import (
    visualize_feature_values_by_pca,
)


@dataclass
class ModelLog:
    start: int = None
    end: int = None
    model: OpenCLIPAttentionViT = None
    mask_config: Dict[int, Tuple[OpenCLIPAttentionViT.ModeOptions, OpenCLIPAttentionViT.MaskOptions]] = None
    attn_out_proj_bias: bool = None
    monitor: Monitor = None
    per_metric_output_dict: OrderedDict[str, np.ndarray[List[torch.Tensor]]] = None
    per_layer_output_dict: List[Dict[str, np.ndarray[torch.Tensor]]] = None

mode = "mask"
mask_type = "X -> T"
mask_layer = 10
end_layer = 12

attention_matrix_monitor_config = OrderedDict({
    "model.visual.transformer.resblocks": OrderedDict({
        "": [
            ("layer_output", Monitor.default_hook_fn),
        ],
        "ln_2": [
            ("intermediate_output", input_hook_fn),
        ],
    })
})

log = ModelLog(start=mask_layer, end=end_layer, mask_config={
    i: (mode, mask_type) if i == mask_layer else ("sink", "X -> X")
    for i in range(mask_layer, ImageFeatures.NUM_LAYERS)
}, attn_out_proj_bias=True)

log.model = OpenCLIPAttentionViT(mask_config=log.mask_config, attn_out_proj_bias=log.attn_out_proj_bias, stop_layer=log.end + 1)
log.monitor = Monitor(log.model, attention_matrix_monitor_config, device=DEVICE)

l10_bias: torch.Tensor = log.model.model.visual.transformer.resblocks[10].attn.out_proj.bias
l10_intermediate_outputs: List[torch.Tensor] = []
ranked_AS_mask: torch.Tensor = torch.load("experiments/saved_masks/ranked_AS_mask50.pt", map_location=OUTPUT_DEVICE)
with torch.no_grad():
    it, max_it, start_vis_it = 1, float("inf"), float("inf")
    convergence = torch.full((batch_size,), False)
    while not torch.all(convergence) and it <= max_it:
        s = "".join("\u25A0" if c else "\u25A1" for c in convergence)    
        print("=" * 160)
        print(f"Iteration {it}: {s}")
        print("=" * 160)
        
        torch.set_default_device(DEVICE)
        log.per_metric_output_dict = log.monitor.reset()
        log.model.load_cache({"mask": (ranked_AS_mask < it), "layer_output": [d["layer_output"] for d in per_layer_output_dict[:log.start]]})
        log.model.forward(images)
        log.per_layer_output_dict = construct_per_layer_output_dict(log.per_metric_output_dict)
        torch.set_default_device(OUTPUT_DEVICE)

        new_MA_mask = (ranked_AS_mask == it)
        
        # SECTION: Visualize the attention matrix before and after
        if it >= start_vis_it:
            print(f"\tNumber of masked MAs: {torch.sum(ranked_AS_mask < 1, dim=1)[VISUALIZED_INDICES].tolist()}")
            print(f"\tNumber of new MAs: {torch.sum(new_MA_mask, dim=1)[VISUALIZED_INDICES].tolist()}")
            
            k_features = ImageFeatures(log.per_layer_output_dict, {}, "default", DEVICE)        
            visualize_feature_norms_per_image(
                features, mask_layer - 1, "layer_output",
                global_cmap=False, write_values=True, cmap="gray",
            )
            for layer_idx in range(mask_layer, end_layer): 
                visualize_feature_norms_per_image(
                    k_features, layer_idx, "intermediate_output",
                    global_cmap=False, write_values=True, cmap="gray",
                )
                visualize_feature_norms_per_image(
                    k_features, layer_idx, "layer_output",
                    global_cmap=False, write_values=True, cmap="gray",
                )
        
        # SECTION: Update the cumulative MA mask with the new attention sinks
        it += 1
        
        # SECTION: Check convergence
        convergence = torch.sum(new_MA_mask, dim=1) == 0
        
        # SECTION: Add L10 intermediates if they are not duplicates
        l10_intermediate_outputs.append(log.per_metric_output_dict["intermediate_output"][mask_layer][-1][~convergence])
        
        # SECTION: Cleanup
        log.per_metric_output_dict = None
        log.per_layer_output_dict = None
            
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
from cuml.manifold import TSNE
test_tsne = torch.tensor(TSNE(n_components=2, n_neighbors=120).fit_transform(torch.cat(l10_intermediate_outputs[:10], dim=0).flatten(0, -2)))

In [ ]:
import matplotlib.colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from cuml.manifold import TSNE

from core.ma_classifier import MAClassifier


tsne2d_fname = "experiments/attention/tsne2d_maybe_better.pt"
if not os.path.exists(tsne2d_fname):
    tsne2d = torch.tensor(TSNE(n_components=2, n_neighbors=120).fit_transform(l10_intermediate_outputs[0][:, ImageFeatures.image_indices].flatten(0, -2)), device=DEVICE)
else:
    tsne2d = test_tsne.view(-1, ImageFeatures.N + 1, 2)[:batch_size].flatten(0, -2).to(DEVICE)
    # tsne2d = torch.load(tsne2d_fname, map_location=DEVICE)

MAC = MAClassifier(model=model.model, start=mask_layer, end=end_layer).to(DEVICE)
mac_norms = MAC.forward(l10_intermediate_outputs[0][:, ImageFeatures.image_indices])
zorder = torch.argsort(mac_norms.flatten())

plt.scatter(*tsne2d.flatten(0, -2)[zorder].mT.numpy(force=True), color=rgb_assignment.flatten(0, -2)[zorder].numpy(force=True))
plt.title("TSNE3d Colors")
plt.show()

sc = plt.scatter(
    *tsne2d.flatten(0, -2)[zorder].mT.numpy(force=True),
    c=mac_norms.flatten()[zorder].numpy(force=True), cmap="magma", norm=matplotlib.colors.LogNorm(),
)
plt.colorbar(sc, cax=make_axes_locatable(plt.gca()).append_axes("right", size="5%", pad=0.05), orientation="vertical")

plt.title("MAC Norms")
plt.show()

In [ ]:
raise Exception()

In [ ]:
%matplotlib inline
from modeling.image_features import ImageFeatures
from visualize.base import (
    construct_per_layer_output_dict,
    visualize_feature_norms_per_image,
)


model_dict: Dict[str, ModelLog] = {
    "train": ModelLog(start=mask_layer, end=detection_layer),
    # "test": ModelLog(start=detection_layer, end=test_layer),
}
for k, log in model_dict.items():
    log.model = OpenCLIPAttentionViT(mode, mask_type, range(log.start, ImageFeatures.NUM_LAYERS), stop_layer=log.end + 1)
    log.monitor = Monitor(log.model, attention_matrix_monitor_config)


n_search = 5
topk_indices = torch.topk(ranked_AS_mask, dim=1, k=n_search + 1).indices

truth_table = torch.full((1 << n_search, n_search), False)
output_table = torch.zeros((1 << n_search, batch_size))

for i, bit_mask in enumerate(itertools.product(*((False, True) for _ in range(n_search)))):
    print(f"Bit mask: {bit_mask}")
    mask = torch.full((batch_size, ImageFeatures.N + 1), False)
    mask[torch.arange(batch_size)[:, None], topk_indices[:, :-1]] = torch.tensor(bit_mask)
    
    with torch.no_grad():
        torch.set_default_device(DEVICE)
        for log in model_dict.values():
            log.per_metric_output_dict = log.monitor.reset()
            log.model.load_cache({"mask": mask, "layer_output": [d["layer_output"] for d in per_layer_output_dict[:log.start]]})
            log.model.forward(images)
            log.per_layer_output_dict = construct_per_layer_output_dict(log.per_metric_output_dict)
        torch.set_default_device(OUTPUT_DEVICE)

    new_features = ImageFeatures(model_dict["train"].per_layer_output_dict, {}, "default", DEVICE)
    # visualize_feature_norms_per_image(
    #     new_features, detection_layer - 1, "layer_output",
    #     global_cmap=False, write_values=True, cmap="gray",
    # )
    feature_norms = torch.norm(new_features.get(layer_idx=detection_layer - 1, key="layer_output", include=(ImageFeatures.CLS, ImageFeatures.IMAGE,), with_batch=True), p=2, dim=-1)
    
    truth_table[i] = torch.tensor(bit_mask)
    output_table[i] = feature_norms[torch.arange(batch_size), topk_indices[:, -1]]
    print(f"\t{output_table[i, 1].item()}")
    
    for log in model_dict.values():
        log.per_metric_output_dict = None
        log.per_layer_output_dict = None
    del new_features, feature_norms
    utils.empty_cache()
    
relevance: torch.Tensor = (output_table.mT @ torch.where(truth_table, 1.0, -1.0)) / (1 << n_search)

In [ ]:
pairwise_suppression = visualize_attention_suppression_per_image(
    features=features,
    layer_idx=10,
    model_weights=model_weights,
    empirical=True,
    normalize=False,
    name_to_mask=mta_masks,
    **plotting_kwargs
)
print(torch.mean(pairwise_suppression, dim=-1)[torch.arange(batch_size)[:, None], topk_indices[:, -1:], topk_indices[:, :-1]][:4])
print(relevance[:4])

In [ ]:
# SECTION: Compare attention sink type artifact tokens vs diagonal type artifact tokens
from visualize.base import visualize_features_per_image
from visualize.attention import visualize_attention_sink_decay
from visualize.projections import visualize_feature_values_by_pca

for layer_idx, attention_weights in attentions["test"].items():
    visualize_attention_sink_decay(
        attention_weights, ranked_AS_mask, mode, f"Layer {layer_idx}: attention sink decay with {mode}ing top k ({mask_type})",
        use_cls_proxy=True, lock_tokens=True
    )

# diagonal_mask = artifact_mask * ~AS_mask
# # torch.save(diagonal_mask, f"{OUTPUT_DIR}/diagonal_mask.pt")
# print(torch.sum(AS_mask).item(), torch.sum(diagonal_mask).item())

# m = torch.full((batch_size, ImageFeatures.N + 1), False)
# m[-4, :] = True
# new_ranked_AS_mask = ranked_AS_mask * m

# visualize_features_per_image(features, 20, "layer_output", mta_mask=AS_mask, highlight=diagonal_mask)
# visualize_feature_values_by_pca(
#     features, 10, "layer_output", {"linear"}, new_ranked_AS_mask, rgb_assignment,
#     ndim=2,
#     with_cls=True,
#     highlight=diagonal_mask,
#     alpha=1.0,
# )